# Catálogo de paneles

Descripción breve de las cuatro tablas que produce el pipeline y de los
datos que contiene cada una.

Para el **qué y por qué** de cada fuente, ver
[`METODOLOGIA.md`](METODOLOGIA.md) (sección 1.2, *Para qué sirve cada
fuente*). Para el **cómo** se generan, ver
[`GUIA_CODIGO.md`](GUIA_CODIGO.md).

Las cifras de este cuaderno se leen de los archivos, así que se
actualizan solas al volver a ejecutarlo.

In [1]:
import warnings

import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.width", 130)
pd.set_option("display.max_columns", 30)

from config_local import DIR_PANEL


def cargar(nombre):
    """Lee un panel, prefiriendo Parquet y cayendo al CSV.

    En el CSV hay que declarar cod_dane como texto: los codigos DANE
    llevan cero a la izquierda en Antioquia (05001) y Atlantico (08001),
    y si pandas los infiere como entero ese cero desaparece.
    """
    try:
        return pd.read_parquet(DIR_PANEL / f"{nombre}.parquet")
    except Exception:
        return pd.read_csv(DIR_PANEL / f"{nombre}.csv",
                           dtype={"cod_dane": "string"})


gfw    = cargar("panel_deforestacion_colombia")
hansen = cargar("panel_hansen")
ideam  = cargar("panel_ideam")
dtd    = cargar("panel_dtd")

print("Paneles cargados.")

Paneles cargados.


## Resumen

Los cuatro paneles cubren la **misma grilla de 5 km** y comparten el mismo
`bosque_base_ha`, el mismo cruce municipal y el mismo indexado de píxel a
celda. Esa base común es lo que permite compararlos entre sí.

Miden conceptos distintos, así que **sus cifras nunca se suman**.

In [2]:
resumen = pd.DataFrame([
    {"panel": "panel_deforestacion_colombia", "fuente": "GFW",
     "mide": "Disturbio de la vegetación", "unidad": "hectáreas",
     "grano": "celda x mes"},
    {"panel": "panel_hansen", "fuente": "Hansen GFC",
     "mide": "Pérdida de cobertura arbórea", "unidad": "hectáreas",
     "grano": "celda x año"},
    {"panel": "panel_ideam", "fuente": "IDEAM / SMByC",
     "mide": "Deforestación de bosque natural", "unidad": "hectáreas",
     "grano": "celda x periodo"},
    {"panel": "panel_dtd", "fuente": "IDEAM / SMByC",
     "mide": "Detecciones de alerta temprana", "unidad": "conteo de puntos",
     "grano": "celda x trimestre"},
]).set_index("panel")

tablas = {"panel_deforestacion_colombia": gfw, "panel_hansen": hansen,
          "panel_ideam": ideam, "panel_dtd": dtd}
resumen["filas"] = [f"{len(tablas[p]):,}" for p in resumen.index]
resumen["columnas"] = [tablas[p].shape[1] for p in resumen.index]
resumen["celdas"] = [f"{tablas[p].cell_id.nunique():,}" for p in resumen.index]
resumen["municipios"] = [tablas[p].cod_dane.nunique() for p in resumen.index]
resumen

,fuente,mide,unidad,grano,filas,columnas,celdas,municipios
panel,,,,,,,,
panel_deforestacion_colombia,GFW,Disturbio de la vegetación,hectáreas,celda x mes,"3,524,664",19,"44,616",1114
panel_hansen,Hansen GFC,Pérdida de cobertura arbórea,hectáreas,celda x año,"267,696",9,"44,616",1114
panel_ideam,IDEAM / SMByC,Deforestación de bosque natural,hectáreas,celda x periodo,"223,080",14,"44,616",1114
panel_dtd,IDEAM / SMByC,Detecciones de alerta temprana,conteo de puntos,celda x trimestre,"1,115,400",14,"44,616",1114


In [3]:
# Verificacion de que la base comun es realmente comun.
celdas = {n: set(t.cell_id) for n, t in tablas.items()}
base = next(iter(celdas.values()))
bosque = {n: t.groupby("cell_id").bosque_base_ha.first() for n, t in tablas.items()}
ref = next(iter(bosque.values()))

print("mismas celdas en los cuatro :", all(c == base for c in celdas.values()))
print("diferencia max en bosque_base_ha:",
      max((v - ref).abs().max() for v in bosque.values()))
print(f"bosque base total           : {ref.sum():,.0f} ha")

mismas celdas en los cuatro : True
diferencia max en bosque_base_ha: 0.0
bosque base total           : 77,713,177 ha


---

## 1. `panel_deforestacion_colombia` — alertas de GFW

**Qué contiene.** Hectáreas con alerta de disturbio de la vegetación por
celda y mes, a partir del producto integrado `gfw_integrated_alerts`
(GLAD-L, GLAD-S2 y RADD), filtrado a confianza alta y muy alta.

**Para qué sirve.** Responde **cuándo** está ocurriendo el fenómeno. Es el
de mayor resolución temporal y el insumo previsto para el modelo.

**Panel balanceado**: toda celda tiene una fila en todo periodo, de modo
que `filas = celdas × periodos` exactamente.

In [4]:
print(f"periodos      : {gfw.periodo.min():%Y-%m} a {gfw.periodo.max():%Y-%m} "
      f"({gfw.periodo.nunique()} meses)")
print(f"balanceado    : {len(gfw) == gfw.cell_id.nunique() * gfw.periodo.nunique()}")
print(f"area con alerta: {gfw.area_def_ha.sum():,.0f} ha")
print(f"celda-mes con evento: {100 * gfw.evento.mean():.2f} %")
print()
print("Columnas:")
for i, c in enumerate(gfw.columns, 1):
    print(f"  {i:2d}. {c:<22} {gfw[c].dtype}")

periodos      : 2020-01 a 2026-07 (79 meses)
balanceado    : True
area con alerta: 1,085,131 ha
celda-mes con evento: 28.01 %

Columnas:
   1. cell_id                object
   2. periodo                datetime64[us]
   3. area_def_ha            float64
   4. lon                    float64
   5. lat                    float64
   6. bosque_base_ha         float64
   7. departamento           object
   8. def_acum_ha            float64
   9. bosque_remanente_ha    float64
  10. tasa_def               float64
  11. evento                 int64
  12. lag_1_ha               float64
  13. lag_2_ha               float64
  14. lag_3_ha               float64
  15. media_movil_3          float64
  16. anio                   int64
  17. mes                    int64
  18. cod_dane               string
  19. municipio              object


In [5]:
gfw[gfw.evento == 1].head(3)

,cell_id,periodo,area_def_ha,lon,lat,bosque_base_ha,departamento,def_acum_ha,bosque_remanente_ha,tasa_def,evento,lag_1_ha,lag_2_ha,lag_3_ha,media_movil_3,anio,mes,cod_dane,municipio
0,-668855_13972,2020-01-01,1.10757,-66.885465,1.397242,1802.5625,Guainía,1.10757,1802.56250,0.000614,1,NaN,NaN,NaN,NaN,2020,1,94885,La Guadalupe
1,-668855_13972,2020-02-01,1.32906,-66.885465,1.397242,1802.5625,Guainía,2.43663,1801.45493,0.000738,1,1.10757,NaN,NaN,NaN,2020,2,94885,La Guadalupe
2,-668855_13972,2020-03-01,0.04924,-66.885465,1.397242,1802.5625,Guainía,2.48587,1800.12587,0.000027,1,1.32906,1.10757,NaN,NaN,2020,3,94885,La Guadalupe


---

## 2. `panel_hansen` — pérdida de cobertura arbórea

**Qué contiene.** Hectáreas que perdieron cobertura arbórea por celda y
año, desde la capa `lossyear` de Hansen Global Forest Change.

**Para qué sirve.** Responde **cómo se compara Colombia con otros
países**: Hansen aplica el mismo algoritmo en todo el planeta. El mismo
producto aporta además la línea base `bosque_base_ha` que usan los cuatro
paneles.

**Ojo con el concepto**: la pérdida de cobertura incluye cosecha de
plantación forestal, incendio y daño natural, así que su cifra es mayor
que la deforestación oficial.

In [6]:
print("Pérdida de cobertura por año (ha):")
print(hansen.groupby("anio").perdida_ha.sum().apply(lambda x: f"{x:,.0f}").to_string())
print()
print("Columnas:")
for i, c in enumerate(hansen.columns, 1):
    print(f"  {i:2d}. {c:<22} {hansen[c].dtype}")

Pérdida de cobertura por año (ha):
anio
2020    324,608
2021    265,200
2022    266,168
2023    197,246
2024    213,785
2025    186,697

Columnas:
   1. cell_id                object
   2. anio                   int64
   3. perdida_ha             float64
   4. lon                    float64
   5. lat                    float64
   6. departamento           object
   7. bosque_base_ha         float64
   8. cod_dane               object
   9. municipio              object


---

## 3. `panel_ideam` — deforestación oficial

**Qué contiene.** Las cinco clases de las capas de cambio del SMByC por
celda y periodo, en hectáreas. La clase de interés es `def_ha`
(deforestación); las demás sirven de contexto y control de calidad.

**Para qué sirve.** Responde **cuánto**, con la cifra oficial de Colombia
y la definición nacional de bosque.

**Dos advertencias sobre esta tabla:**

- `periodo` designa una **transición** entre dos composiciones anuales de
  imágenes, y su cifra corresponde al **año final**: `2022-2023` produce
  el dato oficial de 2023.
- El denominador correcto para sus tasas es **`bosque_ideam_ha`** (bosque
  natural), no `bosque_base_ha` (Hansen, que incluye plantaciones).

In [7]:
col = ["def_ha", "bosque_estable_ha", "regeneracion_ha", "sin_info_ha"]
por_periodo = ideam.groupby("periodo")[col].sum()

# Cifra oficial publicada por el IDEAM, para contrastar.
OFICIAL = {"2020-2021": 174_103, "2021-2022": 123_517,
           "2022-2023": 79_256, "2023-2024": 113_608}
por_periodo["oficial_ha"] = pd.Series(OFICIAL)
por_periodo["recuperado_%"] = (100 * por_periodo.def_ha / por_periodo.oficial_ha).round(2)

# Las hectareas se redondean a entero; el porcentaje conserva decimales
# porque la cercania al 100 % es justamente lo que valida la agregacion.
por_periodo.astype({c: "int64" for c in col + ["oficial_ha"]}, errors="ignore")

,def_ha,bosque_estable_ha,regeneracion_ha,sin_info_ha,oficial_ha,recuperado_%
periodo,,,,,,
2020-2021,173662,59120882,30,116394,174103.0,99.75
2021-2022,123204,58879438,185,147406,123517.0,99.75
2022-2023,78982,58854550,57,561,79256.0,99.65
2023-2024,113422,58745672,324,376,113608.0,99.84
2024-2025,119282,58576834,1069,250,NaN,NaN


In [8]:
print("Los dos denominadores, a escala nacional:")
b = ideam[ideam.periodo == ideam.periodo.min()]
print(f"  bosque_base_ha  (Hansen, dosel >= 30 %) : {b.bosque_base_ha.sum():>14,.0f} ha")
print(f"  bosque_ideam_ha (bosque natural, IDEAM) : {b.bosque_ideam_ha.sum():>14,.0f} ha")
print()
print("Columnas:")
for i, c in enumerate(ideam.columns, 1):
    print(f"  {i:2d}. {c:<22} {ideam[c].dtype}")

Los dos denominadores, a escala nacional:
  bosque_base_ha  (Hansen, dosel >= 30 %) :     77,713,177 ha
  bosque_ideam_ha (bosque natural, IDEAM) :     59,294,546 ha

Columnas:
   1. cell_id                object
   2. periodo                object
   3. bosque_estable_ha      float64
   4. def_ha                 float64
   5. sin_info_ha            float64
   6. regeneracion_ha        float64
   7. no_bosque_ha           float64
   8. bosque_ideam_ha        float64
   9. lon                    float64
  10. lat                    float64
  11. departamento           object
  12. bosque_base_ha         float64
  13. cod_dane               object
  14. municipio              object


---

## 4. `panel_dtd` — alertas tempranas oficiales

**Qué contiene.** Número de puntos de alerta que el IDEAM publicó por
celda y trimestre, más cuántos de ellos caen en un área protegida del
SINAP.

**Para qué sirve.** Responde **dónde**, con respaldo de la autoridad
ambiental. Es el único producto oficial de cadencia sub-anual disponible
para Colombia.

**La unidad es el conteo, no la hectárea.** Los datos son puntos sin
superficie asociada: marcan un sitio donde se detectó un cambio
compatible con deforestación, sin cuantificar su extensión.

**Ningún agregado de esta tabla sirve como serie temporal.** El criterio
de detección cambió a lo largo de la serie, así que los conteos por año
se mueven en sentido contrario a la deforestación real. La tabla sirve
**dentro** de cada trimestre, para comparar unos lugares con otros.

In [9]:
print(f"trimestres: {dtd.trimestre_txt.min()} a {dtd.trimestre_txt.max()} "
      f"({dtd.trimestre_txt.nunique()})")
print(f"balanceado: {len(dtd) == dtd.cell_id.nunique() * dtd.trimestre_txt.nunique()}")
print()
print("Detecciones por año:")
print(dtd.groupby("anio").n_alertas.sum().apply(lambda x: f"{x:,}").to_string())
print()

# sinap_disponible marca los trimestres en que el IDEAM informa el campo:
# un vacio de informacion y una ausencia de alertas son cosas distintas.
con = dtd[dtd.sinap_disponible]
print(f"En areas protegidas del SINAP: {int(con.n_alertas_sinap.sum()):,} de "
      f"{con.n_alertas.sum():,} ({100 * con.n_alertas_sinap.sum() / con.n_alertas.sum():.1f} %), "
      f"en {con.trimestre_txt.nunique()} de {dtd.trimestre_txt.nunique()} trimestres")
print()
print("Columnas:")
for i, c in enumerate(dtd.columns, 1):
    print(f"  {i:2d}. {c:<22} {dtd[c].dtype}")

trimestres: 2020-T1 a 2026-T1 (25)
balanceado: True

Detecciones por año:


anio
2020    21,895
2021    15,065
2022    27,579
2023    31,923
2024    45,754
2025    51,072
2026    28,990



En areas protegidas del SINAP: 19,645 de 177,215 (11.1 %), en 18 de 25 trimestres

Columnas:
   1. cell_id                object
   2. periodo                datetime64[us]
   3. trimestre_txt          object
   4. anio                   int64
   5. trimestre              int64
   6. n_alertas              int64
   7. n_alertas_sinap        float64
   8. sinap_disponible       bool
   9. lon                    float64
  10. lat                    float64
  11. departamento           object
  12. bosque_base_ha         float64
  13. cod_dane               object
  14. municipio              object


---

## Las tres fuentes en hectáreas, lado a lado

Las tres primeras tablas miden superficie y son comparables en magnitud.
Las razones frente a la cifra oficial **varían año a año**, así que
convertir una fuente en otra exige un factor específico para cada año.

In [10]:
anual = pd.DataFrame({
    "GFW_ha": gfw.groupby("anio").area_def_ha.sum(),
    "Hansen_ha": hansen.groupby("anio").perdida_ha.sum(),
    "IDEAM_ha": (ideam.assign(anio=ideam.periodo.str[-4:].astype(int))
                      .groupby("anio").def_ha.sum()),
})
anual["oficial_ha"] = pd.Series(
    {2021: 174_103, 2022: 123_517, 2023: 79_256, 2024: 113_608})
anual["Hansen/IDEAM"] = (anual.Hansen_ha / anual.IDEAM_ha).round(2)
anual["GFW/IDEAM"] = (anual.GFW_ha / anual.IDEAM_ha).round(2)

for c in ["GFW_ha", "Hansen_ha", "IDEAM_ha", "oficial_ha"]:
    anual[c] = anual[c].round(0)
anual

,GFW_ha,Hansen_ha,IDEAM_ha,oficial_ha,Hansen/IDEAM,GFW/IDEAM
anio,,,,,,
2020,215345.0,324608.0,NaN,NaN,NaN,NaN
2021,195833.0,265200.0,173663.0,174103.0,1.53,1.13
2022,172082.0,266168.0,123204.0,123517.0,2.16,1.40
2023,122362.0,197246.0,78982.0,79256.0,2.50,1.55
2024,156640.0,213785.0,113423.0,113608.0,1.88,1.38
2025,144843.0,186697.0,119282.0,NaN,1.57,1.21
2026,78026.0,NaN,NaN,NaN,NaN,NaN


---

## Cómo cruzarlos

Las cuatro tablas comparten `cell_id` y `cod_dane`, de modo que el cruce
es directo. Al agregar por municipio conviene usar `cod_dane` como llave,
nunca el nombre: las tildes y mayúsculas no son consistentes entre
fuentes.

In [11]:
COD = "18150"   # Cartagena del Chaira, Caqueta

print(f"Municipio {COD} — {ideam[ideam.cod_dane == COD].municipio.iloc[0]}")
print(f"  GFW    (disturbio, ha)      : {gfw[gfw.cod_dane == COD].area_def_ha.sum():>10,.0f}")
print(f"  Hansen (pérdida, ha)        : {hansen[hansen.cod_dane == COD].perdida_ha.sum():>10,.0f}")
print(f"  IDEAM  (deforestación, ha)  : {ideam[ideam.cod_dane == COD].def_ha.sum():>10,.0f}")
print(f"  DTD    (detecciones)        : {dtd[dtd.cod_dane == COD].n_alertas.sum():>10,}")

Municipio 18150 — Cartagena Del Chairá


  GFW    (disturbio, ha)      :     87,861
  Hansen (pérdida, ha)        :     80,968
  IDEAM  (deforestación, ha)  :     55,725
  DTD    (detecciones)        :     19,138


In [12]:
# Ranking municipal segun la fuente oficial, que es el uso previsto
# para el tamizaje de riesgo.
(ideam.groupby(["cod_dane", "municipio", "departamento"])
      .def_ha.sum().sort_values(ascending=False)
      .head(10).round(0).reset_index())

,cod_dane,municipio,departamento,def_ha
0,18150,Cartagena Del Chairá,Caquetá,55725.0
1,18753,San Vicente Del Caguán,Caquetá,44512.0
2,50325,Mapiripán,Meta,34891.0
3,50350,La Macarena,Meta,32950.0
4,95015,Calamar,Guaviare,30143.0
5,95001,San José Del Guaviare,Guaviare,29724.0
6,18756,Solano,Caquetá,21368.0
7,86571,Puerto Guzmán,Putumayo,19866.0
8,54810,Tibú,Norte De Santander,18858.0
9,95025,El Retorno,Guaviare,17710.0
